# Send Photos and Videos to Gemini Without Converting Them by Hand

In this video, I'm going to pass a photo column and a video column straight into a Gemini call in Pixeltable, a multimodal backend whose columns can call models.

With Pydantic AI, our code builds a message part for each media file in each call, such as its bytes plus a media type. In Pixeltable, we pass the column itself.

A video over 3 MiB even goes through Gemini's Files API, and Pixeltable deletes the upload after the call.

We'll start with Pydantic AI, then send photos and videos from Pixeltable.

## Your Gemini key goes in the environment

The notebook is in the `media-to-llm` folder of the repo linked below. Start Jupyter there with `uv run jupyter lab`. Off camera, I set `GEMINI_API_KEY` in my environment, where Pydantic AI and Pixeltable both read it.

In [ ]:
!uv sync

In [ ]:
import pixeltable as pxt
import pixeltable.functions as pxtf

# Start fresh: drop the tables from any earlier run.
pxt.drop_dir('media_demo', force=True, if_not_exists='ignore')

## With Pydantic AI, our code reads the bytes and states the media type

The Pydantic AI docs send an image as `BinaryContent`: the raw bytes, plus a media type written in the code. I'll send one desk lamp photo this way, from the sample photos of a secondhand marketplace app.

In [ ]:
import httpx
from pydantic_ai import Agent, BinaryContent

BASE = 'https://raw.githubusercontent.com/apreshill/pixeltable-marketplace/main/samples/'
PROMPT = 'Describe the item in this photo in one sentence, for a secondhand listing.'

photo = BinaryContent(
    data=httpx.get(BASE + 'desk-lamp-1.jpg').content,  # our code reads the bytes
    media_type='image/jpeg',  # and states the media type
)
agent = Agent('google:gemini-3.8-flash')
result = await agent.run([PROMPT, photo])
result.output, result.usage.input_tokens

That works, and our code builds a part like this for each media file in each call. Gemini counted 1,105 input tokens, and we'll check that number again in Pixeltable.

## The photo column goes straight into the Gemini call

`schema.py` declares the `photos` table. The only columns you fill in are `name` and `photo`. `response` is a computed column that calls Gemini on every new row, and `answer` pulls the text out of the response.

In [ ]:
!sed -n '8,17p' schema.py

The list holds the same prompt and the `photo` column itself. Because `photo` is an image column, Pixeltable loads it as an image, not a string, and the google-genai SDK sends it as `image/jpeg` bytes.

`pxt schema diff` shows the plan, and `pxt schema update` creates the table.

In [ ]:
!pxt schema diff schema.py media_demo

In [ ]:
!pxt schema update schema.py media_demo -f

## Each photo gets its answer when it's inserted

I'll insert four photos by URL. Pixeltable downloads each one when it inserts the row, and Gemini answers during the same insert.

In [ ]:
NAMES = ['desk-lamp-1.jpg', 'film-camera-1.jpg', 'cruiser-bike-1.jpg', 'guitar-1.jpg']

photos = pxt.get_table('media_demo.photos')
photos.insert([{'name': name, 'photo': BASE + name} for name in NAMES])

In [ ]:
photos.select(
    photos.name, photos.photo, photos.answer, input_tokens=photos.response.usage_metadata.prompt_token_count
).order_by(photos.name).collect()

Each photo sits in its row next to Gemini's answer. Every photo used 1,105 input tokens, the same as the Pydantic AI call: the prompt plus one image.

## A video column goes into the call the same way

Gemini takes a video inside the request, or as a media file uploaded to its Files API first. Pixeltable picks by size: up to 3 MiB goes inside the request, and a bigger video goes through the Files API.

`schema_video.py` repeats `photos` and adds `videos`. Because `video` is a video column, the call gets the downloaded media file, not the URL.

In [ ]:
!sed -n '10p;21,26p' schema_video.py

In [ ]:
!pxt schema diff schema_video.py media_demo

`photos` shows no change, so its answers stay as they are, and `videos` is new.

In [ ]:
!pxt schema update schema_video.py media_demo -f

## A video over 3 MiB goes through the Files API and is deleted after the call

To see which path each video takes, I'll import a small module that prints a line for each Files API upload and delete. It's only for this demo.

Then I'll insert the same 18-second street video from the Pixeltable repo at two sizes.

In [ ]:
import watch_files  # demo only: prints each Files API upload and delete

VIDEO_BASE = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/tests/data/'

videos = pxt.get_table('media_demo.videos')
videos.insert(
    [
        {'name': 'bangkok_half_res.mp4', 'video': VIDEO_BASE + 'videos/bangkok_half_res.mp4'},
        {'name': 'bangkok.mp4', 'video': VIDEO_BASE + 'orig_videos/bangkok.mp4'},
    ]
)

The module printed lines for only one video. Pixeltable uploaded it before the call and deleted it after, so the upload doesn't stay in your Gemini project. The `size_mib` column shows which video that was.

In [ ]:
size_mib = (videos.video.get_metadata().size.astype(pxt.Float) / 2**20).round(1)
videos.select(videos.name, size_mib=size_mib, answer=videos.answer).collect()

The 6.1 MiB video went through the Files API, and the 2.1 MiB video went inside the request.

In Pixeltable 0.7.25, this works for images and videos, not for audio or PDFs.

## Two columns went to Gemini with no conversion code

We sent the same desk lamp photo two ways. With Pydantic AI, our code read the bytes and wrote `image/jpeg`. In Pixeltable, the `photo` column went straight into `generate_content`.

The `video` column went in the same way, and Pixeltable picked the path for each video by its size.

If this helps, a star on Pixeltable's own GitHub repo helps other developers find it.